In [1]:
import numpy as np
import pandas as pd

FEATURES = [
    "temperature_2m", "weather_code", "apparent_temperature",
    "precipitation", "relative_humidity_2m", "dew_point_2m",
    "surface_pressure", "wind_speed_10m", "wind_direction_10m"
]

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("date").reset_index(drop=True)

    # --- Time features ---
    df["hour"]        = df["date"].dt.hour
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["month"]       = df["date"].dt.month
    df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)

    # --- Cyclical encoding ---
    # Converts hour 23 → close to hour 0 mathematically
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"]   = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"]   = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"]     = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["doy_cos"]     = np.cos(2 * np.pi * df["day_of_year"] / 365)

    # --- Lag features (most important for time-series) ---
    for col in FEATURES:
        df[f"{col}_lag_1h"]  = df[col].shift(1)
        df[f"{col}_lag_3h"]  = df[col].shift(3)
        df[f"{col}_lag_6h"]  = df[col].shift(6)
        df[f"{col}_lag_24h"] = df[col].shift(24)   # same hour yesterday
        df[f"{col}_lag_48h"] = df[col].shift(48)   # same hour 2 days ago

    # --- Rolling statistics ---
    for col in ["temperature_2m", "precipitation", "wind_speed_10m", "relative_humidity_2m"]:
        df[f"{col}_roll_3h"]  = df[col].rolling(3,  min_periods=1).mean()
        df[f"{col}_roll_6h"]  = df[col].rolling(6,  min_periods=1).mean()
        df[f"{col}_roll_24h"] = df[col].rolling(24, min_periods=1).mean()
        df[f"{col}_std_6h"]   = df[col].rolling(6,  min_periods=1).std()

    # --- Drop rows with NaN from initial lags ---
    df.dropna(inplace=True)
    return df

In [6]:
import pandas as pd

df = pd.read_csv("/home/makos/dev/weather_dag/raw_data/two_years.csv",index_col=0)

In [8]:
df

,date,temperature_2m,weather_code,apparent_temperature,precipitation,relative_humidity_2m,dew_point_2m,surface_pressure,wind_speed_10m,wind_direction_10m
0,2024-04-12 00:00:00+00:00,14.659500,3.0,12.770615,0.0,55.753033,5.9095,910.64520,5.959060,115.016870
1,2024-04-12 01:00:00+00:00,15.109500,3.0,13.108503,0.0,48.455875,4.3095,910.34160,4.394360,145.007900
2,2024-04-12 02:00:00+00:00,15.259500,3.0,12.774542,0.0,40.646248,1.9595,910.11980,4.680000,157.380100
3,2024-04-12 03:00:00+00:00,14.709500,3.0,12.161451,0.0,36.998917,0.1595,909.76050,3.075841,159.443880
4,2024-04-12 04:00:00+00:00,11.009500,3.0,8.047935,0.0,52.504630,1.6595,908.89890,7.594208,328.570500
...,...,...,...,...,...,...,...,...,...,...
17539,2026-04-12 19:00:00+00:00,18.459501,0.0,16.102379,0.0,36.905570,3.4595,920.98360,5.683555,10.954029
17540,2026-04-12 20:00:00+00:00,18.209501,0.0,16.044323,0.0,36.444595,3.0595,920.90295,3.847960,10.784258
17541,2026-04-12 21:00:00+00:00,16.909500,0.0,15.989340,0.0,51.193153,6.7595,920.66240,0.648999,146.309900
17542,2026-04-12 22:00:00+00:00,14.709500,0.0,12.969009,0.0,50.069070,4.4095,920.12225,2.747581,148.392550


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17544 entries, 0 to 17543
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  17544 non-null  str    
 1   temperature_2m        17544 non-null  float64
 2   weather_code          17544 non-null  float64
 3   apparent_temperature  17544 non-null  float64
 4   precipitation         17544 non-null  float64
 5   relative_humidity_2m  17544 non-null  float64
 6   dew_point_2m          17544 non-null  float64
 7   surface_pressure      17544 non-null  float64
 8   wind_speed_10m        17544 non-null  float64
 9   wind_direction_10m    17544 non-null  float64
dtypes: float64(9), str(1)
memory usage: 1.3 MB


In [11]:
df["date"] = pd.to_datetime(df["date"])

In [12]:
df_prepocessed = engineer_features(df)

In [13]:
df_prepocessed

,date,temperature_2m,weather_code,apparent_temperature,precipitation,relative_humidity_2m,dew_point_2m,surface_pressure,wind_speed_10m,wind_direction_10m,...,precipitation_roll_24h,precipitation_std_6h,wind_speed_10m_roll_3h,wind_speed_10m_roll_6h,wind_speed_10m_roll_24h,wind_speed_10m_std_6h,relative_humidity_2m_roll_3h,relative_humidity_2m_roll_6h,relative_humidity_2m_roll_24h,relative_humidity_2m_std_6h
48,2024-04-14 00:00:00+00:00,11.759500,3.0,10.717299,0.0,78.057710,8.0595,910.40910,3.671294,101.309900,...,0.008333,0.08165,3.162938,2.908763,3.980540,0.564446,77.271357,73.763888,81.031102,6.729366
49,2024-04-14 01:00:00+00:00,12.209500,3.0,10.925562,0.0,75.521110,8.0095,910.28894,5.241679,105.945465,...,0.008333,0.08165,3.996271,3.432519,4.165403,0.973045,76.729707,75.567243,80.302542,5.056893
50,2024-04-14 02:00:00+00:00,12.059500,3.0,10.888635,0.0,76.011246,7.9595,910.41900,4.379589,99.462250,...,0.008333,0.08165,4.430854,3.609278,4.272885,1.042179,76.530022,77.272435,79.403053,1.721832
51,2024-04-14 03:00:00+00:00,11.709500,3.0,10.643064,0.0,76.733795,7.7595,910.21250,3.319036,49.398785,...,0.008333,0.08165,4.313435,3.738186,4.330401,0.925611,76.088717,76.680037,78.477009,0.884681
52,2024-04-14 04:00:00+00:00,10.159500,3.0,8.813246,0.0,82.200250,7.2595,909.69336,4.394360,34.992100,...,0.008333,0.00000,4.030995,4.013633,4.412876,0.808219,78.315097,77.522402,77.807664,2.445870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17539,2026-04-12 19:00:00+00:00,18.459501,0.0,16.102379,0.0,36.905570,3.4595,920.98360,5.683555,10.954029,...,0.000000,0.00000,4.660382,4.878866,4.403006,0.979221,37.405877,38.393269,57.992549,1.982493
17540,2026-04-12 20:00:00+00:00,18.209501,0.0,16.044323,0.0,36.444595,3.0595,920.90295,3.847960,10.784258,...,0.000000,0.00000,4.283517,4.501518,4.430650,0.834510,36.916400,37.412197,57.264998,0.658447
17541,2026-04-12 21:00:00+00:00,16.909500,0.0,15.989340,0.0,51.193153,6.7595,920.66240,0.648999,146.309900,...,0.000000,0.00000,3.393505,3.829684,4.420191,1.765441,41.514439,39.569415,56.548064,5.717680
17542,2026-04-12 22:00:00+00:00,14.709500,0.0,12.969009,0.0,50.069070,4.4095,920.12225,2.747581,148.392550,...,0.000000,0.00000,2.414847,3.537614,4.449161,1.777285,45.902273,41.654075,55.906407,6.979885


In [14]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib
import numpy as np

TARGET_COLS  = ["temperature_2m", "apparent_temperature", "precipitation",
                "wind_speed_10m", "wind_direction_10m", "relative_humidity_2m"]

def build_datasets(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in TARGET_COLS + ["date"]]
    split_date   = df["date"].max() - pd.Timedelta(days=30)

    train = df[df["date"] <  split_date]
    test  = df[df["date"] >= split_date]

    return (
        train[feature_cols], train[TARGET_COLS],
        test[feature_cols],  test[TARGET_COLS],
        feature_cols
    )


def train_model(X_train, y_train):
    model = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    )
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test) -> dict:
    preds = model.predict(X_test)
    metrics = {}
    for i, col in enumerate(TARGET_COLS):
        mae  = mean_absolute_error(y_test[col], preds[:, i])
        rmse = np.sqrt(mean_squared_error(y_test[col], preds[:, i]))
        metrics[f"mae_{col}"]  = mae
        metrics[f"rmse_{col}"] = rmse
    return metrics

In [15]:
X_train, y_train, X_test, y_test, features = build_datasets(df_prepocessed)

In [16]:
model = train_model(X_train=X_train, y_train=y_train)

In [17]:
evaluate_model(model, X_test, y_test)

{'mae_temperature_2m': 0.36816729581007246,
 'rmse_temperature_2m': np.float64(0.48600888181176977),
 'mae_apparent_temperature': 0.4599752469513208,
 'rmse_apparent_temperature': np.float64(0.608438899638691),
 'mae_precipitation': 0.00619996804144874,
 'rmse_precipitation': np.float64(0.03612622851578685),
 'mae_wind_speed_10m': 0.6211978761103627,
 'rmse_wind_speed_10m': np.float64(0.8281195310839081),
 'mae_wind_direction_10m': 44.56760776572923,
 'rmse_wind_direction_10m': np.float64(73.16852084757099),
 'mae_relative_humidity_2m': 1.6583048693867974,
 'rmse_relative_humidity_2m': np.float64(2.290347397122956)}